# 1. Setup, Dependencies & Reproducibility
Install required packages, import core libraries, set random seeds for reproducible runs, and select compute device.

In [2]:
import os
import sys
import random
import copy
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

# Add project root to path for src imports
repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '../..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.utils import seed_everything, load_config
seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device configured: {device} | Seed: {seed_val}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 106.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.5 MB/s eta 0:00:00
Device configured: cuda


# 2. Specialized VSFC Ontology Knowledge Graph Engine
Initializes `VSFCOntologyEngine` tailored for 3-class sentiment analysis (Negative, Neutral, Positive) via deep semantic inference and polarity injection.

In [3]:
# Initialize VSFCOntologyEngine using relative path to ontology RDF
from src.ontology_engine import OntologyEngine, VSFCOntologyEngine
from src.utils import load_config

config_path = os.path.join(repo_root, 'configs/vsfc.yaml')
config = load_config(config_path)
onto_cfg = config.get('ontology', {})

ontology_path = os.path.join(repo_root, onto_cfg.get('rdf_path', 'ontology/ekman_appraisal_ontology.rdf'))

ontengine = VSFCOntologyEngine(
    rdf_path=ontology_path,
    default_conf=onto_cfg.get('default_conf', 0.9),
    negation_attenuation=onto_cfg.get('negation_attenuation', 0.4),
    alpha_similarity=onto_cfg.get('alpha_similarity', 0.2),
    verbose=True,
)


Loading RDF knowledge graph from: /kaggle/input/datasets/minkduck/kg-ekman6/ekman6_4_9.rdf
Loading Knowledge Graph for VSFC: /kaggle/input/datasets/minkduck/kg-ekman6/ekman6_4_9.rdf
Loaded 91005 triples.


# 3. Vector Distribution Inspection
Inspects 24-dimensional continuous feature vectors across representative test sentences and validates activation patterns.

In [4]:
COLS_24D = (
    ["Anger", "Disgust", "Fear", "Happiness", "Neutral", "Sadness", "Surprise"]
    + [
        "Pleas",
        "Unpleas",
        "Loss",
        "Danger",
        "Unpred",
        "Sudden",
        "GoalObs",
        "Unfair",
        "Other",
    ]
    + ["Int_Hi", "Int_Me", "Int_Lo"]
    + ["Pos", "Neg", "Neu"]
    + ["NegHit", "NegCnt"]
)


def inspect_vectors(texts, engine):
    """Build a structured inspection DataFrame for ontology feature vectors."""
    data = []
    for txt in texts:
        vec = engine.getvector(txt)
        if isinstance(vec, tuple):
            vec = vec[0]
        vec_rounded = np.round(vec, 2)
        row = {"Sentence": txt}
        for i, val in enumerate(vec_rounded):
            row[COLS_24D[i]] = val
        data.append(row)
    return pd.DataFrame(data)


sample_texts = [
    "Món này ngon tuyệt vời",
    "Thái độ phục vụ quá tệ hại",
    "Không ngon lắm",
    "Món ăn bình thường, không có gì đặc sắc",
    "Tôi rất thất vọng về cách làm việc",
    "Sợ quá đi mất",
    "Tôi chẳng thấy buồn cười gì cả",
]

df_vector_preview = inspect_vectors(sample_texts, ontengine)
display(
    df_vector_preview[
        ["Sentence", "Happiness", "Sadness", "Pos", "Neg", "Neu", "NegHit"]
    ]
)

,Sentence,Happiness,Sadness,Pos,Neg,Neu,NegHit
0,Món này ngon tuyệt vời,0.60,0.00,0.94,0.00,0.00,0.0
1,Thái độ phục vụ quá tệ hại,0.00,0.00,0.00,0.00,0.00,0.0
2,Không ngon lắm,0.00,0.00,0.00,0.00,0.00,1.0
3,"Món ăn bình thường, không có gì đặc sắc",0.00,0.00,0.00,0.00,0.74,1.0
4,Tôi rất thất vọng về cách làm việc,0.42,0.85,0.81,1.00,0.00,0.0
5,Sợ quá đi mất,0.00,0.00,0.00,0.98,0.00,0.0
6,Tôi chẳng thấy buồn cười gì cả,0.00,0.00,0.00,0.00,0.00,0.0


# 4. Dataset Loading & Preprocessing (UIT-VSFC)
Reads UIT-VSFC sentences and sentiment labels (0: Negative, 1: Neutral, 2: Positive), constructs PyTorch DataLoaders, and computes balanced class weights.

In [5]:
# 1. Trích xuất vector cảm xúc thô VnEmoLex (Doãn & Lưu, 2022)
# Chỉ trích xuất và cộng dồn điểm cảm xúc của từ vựng theo từ điển, không dùng appraisal, polarity, negation rules hay similarity matrix
def get_vnemolex_vector(text, engine):
    tokens = ViTokenizer.tokenize(str(text).lower()).split()
    flat_tokens = [engine._norm(t) for t in tokens if engine._norm(t)]
    v_emo = np.zeros(len(engine.ALLEMOTIONS), dtype=np.float32)

    idx, n = 0, len(flat_tokens)
    while idx < n:
        matched_entry, matched_len = None, 0
        for l_span in range(min(engine.mwe_max_len, n - idx), 0, -1):
            phrase = " ".join(flat_tokens[idx : idx + l_span])
            if phrase in engine.lexicon_map:
                matched_entry = engine.lexicon_map[phrase]
                matched_len = l_span
                break

        if not matched_entry:
            idx += 1
            continue

        for ent in matched_entry:
            if ent.get("isnegation", False):
                continue
            score = ent.get("score", 1.0)
            for e, w in ent.get("emotions", {}).items():
                if e in engine.ALLEMOTIONS:
                    v_emo[engine.ALLEMOTIONS.index(e)] += score * w

        idx += matched_len

    return v_emo  # Vector cảm xúc thô 7 chiều


# 2. Khởi tạo Tokenizers cho cả 2 backbone
tokenizer_phobert = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")
tokenizer_visobert = AutoTokenizer.from_pretrained("uitnlp/visobert")


class DualBackboneDataset(Dataset):

    def __init__(self, df, tokenizer, engine, is_phobert=False, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.engine = engine
        self.is_phobert = is_phobert
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        txt = str(self.df.iloc[idx]["Sentence"])
        lbl = int(self.df.iloc[idx]["Sentiment"])

        # PhoBERT bắt buộc tiền xử lý tách từ bằng pyvi trước khi mã hoá BPE
        input_text = ViTokenizer.tokenize(txt) if self.is_phobert else txt

        enc = self.tokenizer(
            input_text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        vnemolex_vec = get_vnemolex_vector(txt, self.engine)
        ont_vec = self.engine.getvector(txt)
        if isinstance(ont_vec, tuple):
            ont_vec = ont_vec[0]

        return {
            "input_ids": enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
            "lexicon_features": torch.tensor(vnemolex_vec, dtype=torch.float),
            "ontology_features": torch.tensor(ont_vec, dtype=torch.float),
            "labels": torch.tensor(lbl, dtype=torch.long),
        }


# 3. Load UIT-VSFC từ các file txt
BASE_PATH = os.path.join(repo_root, "data", "vsfc", "UIT-VSFC")
if not os.path.exists(BASE_PATH):
    BASE_PATH = os.path.join(repo_root, "data", "vsfc")


def load_vsfc_folder(folder_name):
    sent_path = f"{BASE_PATH}/{folder_name}/sents.txt"
    label_path = f"{BASE_PATH}/{folder_name}/sentiments.txt"
    with open(sent_path, "r", encoding="utf-8") as f:
        sentences = [line.strip() for line in f.readlines()]
    with open(label_path, "r", encoding="utf-8") as f:
        labels = [int(line.strip()) for line in f.readlines()]
    min_len = min(len(sentences), len(labels))
    return pd.DataFrame(
        {"Sentence": sentences[:min_len], "Sentiment": labels[:min_len]}
    )


df_train = load_vsfc_folder("train")
df_valid = load_vsfc_folder("dev")
df_test = load_vsfc_folder("test")

print(f"Loaded UIT-VSFC: Train={len(df_train)}, Dev={len(df_valid)}, Test={len(df_test)}")

# Dataloaders cho PhoBERT
BATCH_SIZE = 32
train_loader_pho = DataLoader(
    DualBackboneDataset(df_train, tokenizer_phobert, ontengine, is_phobert=True),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)
valid_loader_pho = DataLoader(
    DualBackboneDataset(df_valid, tokenizer_phobert, ontengine, is_phobert=True),
    batch_size=BATCH_SIZE,
)
test_loader_pho = DataLoader(
    DualBackboneDataset(df_test, tokenizer_phobert, ontengine, is_phobert=True),
    batch_size=BATCH_SIZE,
)

# Dataloaders cho ViSoBERT
train_loader_viso = DataLoader(
    DualBackboneDataset(df_train, tokenizer_visobert, ontengine, is_phobert=False),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)
valid_loader_viso = DataLoader(
    DualBackboneDataset(df_valid, tokenizer_visobert, ontengine, is_phobert=False),
    batch_size=BATCH_SIZE,
)
test_loader_viso = DataLoader(
    DualBackboneDataset(df_test, tokenizer_visobert, ontengine, is_phobert=False),
    batch_size=BATCH_SIZE,
)

# Balanced Class Weights cho 3 lớp (0: Neg, 1: Neu, 2: Pos)
train_labels = df_train["Sentiment"].tolist()
cls_w = compute_class_weight(
    "balanced", classes=np.unique(train_labels), y=train_labels
)
weights_tensor = torch.tensor(cls_w, dtype=torch.float).to(device)
print(f"Class Weights [Neg, Neu, Pos]: {weights_tensor.cpu().numpy()}")


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/471k [00:00<?, ?B/s]

Loaded UIT-VSFC: Train=11426, Dev=1583, Test=3166
Class Weights [Neg, Neu, Pos]: [0.71524256 8.315866   0.6749365 ]


# 5. Neural-Symbolic Model Architectures
Defines neural-symbolic architectures for 3-class sentiment analysis:
1. `ViSoBERT_Fusion`: Standard ViSoBERT backbone supporting baseline, concatenation, and gated projection.
2. `ViSoBERT_DeepOntology_V2`: Multi-layer bottleneck projection with LayerNorm and Residual Gating.
3. `OntologyConceptProjector` & `ViSoBERT_Ontology_CrossAttention` (OCA): Token-level multi-head cross-attention over projected concept tokens.

In [6]:
# Import lexicon vs ontology comparative models from src.models
from src.models.lexicon_ontology_models import (
    Model_Baseline,
    Model_VnEmoLex_Concat,
    Model_Ontology_RawGate,
)

VNEMOLEX_DIM = 7
ONTOLOGY_DIM = 24
print('✅ Models loaded from src.models.lexicon_ontology_models: Model_Baseline, Model_VnEmoLex_Concat, Model_Ontology_RawGate')


# 6. Training & Evaluation Pipeline
Provides training utilities with validation monitoring, early stopping, and checkpoint restoration.

In [7]:
def evaluate(model, loader, feature_key="lexicon_features"):
    """Evaluate accuracy and macro F1 score on a given DataLoader."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            feat = batch[feature_key].to(device) if feature_key in batch else None

            outputs = model(ids, mask, feat)
            _, pr = torch.max(outputs, 1)
            preds.extend(pr.cpu().numpy())
            targets.extend(labels.cpu().numpy())

    return accuracy_score(targets, preds), f1_score(
        targets, preds, average="macro"
    )


def train_experiment(
    model,
    name,
    train_loader,
    valid_loader,
    feature_key="lexicon_features",
    epochs=15,
    patience=3,
):
    """Execute training loop with class-weighted cross-entropy and early stopping."""
    print(f"\nTraining Model: {name}")
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=2e-5, weight_decay=0.01
    )
    criterion = nn.CrossEntropyLoss(weight=weights_tensor)

    best_f1 = 0.0
    patience_counter = 0
    best_weights = copy.deepcopy(
        model.module.state_dict()
        if isinstance(model, nn.DataParallel)
        else model.state_dict()
    )

    for ep in range(epochs):
        model.train()
        loss_acc = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            feat = (
                batch[feature_key].to(device) if feature_key in batch else None
            )

            outputs = model(ids, mask, feat)
            loss = criterion(outputs, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            loss_acc += loss.item()

        acc, f1 = evaluate(model, valid_loader, feature_key=feature_key)
        avg_loss = loss_acc / len(train_loader)
        print(
            f"Epoch {ep + 1:02d} | Loss: {avg_loss:.4f} | Val F1: {f1:.4f} | Val Acc: {acc:.4f}",
            end=" ",
        )

        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0
            best_weights = copy.deepcopy(
                model.module.state_dict()
                if isinstance(model, nn.DataParallel)
                else model.state_dict()
            )
            print("(New Best)")
        else:
            patience_counter += 1
            print(f"(Patience: {patience_counter}/{patience})")

        if patience_counter >= patience:
            print(f"Early stopping triggered after {ep + 1} epochs.")
            break

    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(best_weights)
    else:
        model.load_state_dict(best_weights)

    return model

# 7. Model Training Across Configurations
Trains all comparative baseline and neural-symbolic models, including the proposed Ontology-Guided Cross-Attention architecture (OCA).

In [8]:
CLASS_NAMES = ["Negative", "Neutral", "Positive"]
NUM_CLASSES = len(CLASS_NAMES)
PHO_ID = "vinai/phobert-base-v2"
VISO_ID = "uitnlp/visobert"

# ==================== NHÓM 1: PHOBERT ====================
# 1. PhoBERT Baseline
m_pho_base = Model_Baseline(NUM_CLASSES, model_name=PHO_ID)
m_pho_base = train_experiment(
    m_pho_base,
    "1. PhoBERT Baseline",
    train_loader_pho,
    valid_loader_pho,
    feature_key="lexicon_features",
    patience=3,
)

# 2. PhoBERT + VnEmoLex Concat (Doãn & Lưu, 2022)
m_pho_lex = Model_VnEmoLex_Concat(NUM_CLASSES, model_name=PHO_ID)
m_pho_lex = train_experiment(
    m_pho_lex,
    "2. PhoBERT + VnEmoLex (Doãn & Lưu, 2022)",
    train_loader_pho,
    valid_loader_pho,
    feature_key="lexicon_features",
    patience=3,
)

# 3. PhoBERT + Full Ontology (RawGate)
m_pho_ont = Model_Ontology_RawGate(NUM_CLASSES, model_name=PHO_ID)
m_pho_ont = train_experiment(
    m_pho_ont,
    "3. PhoBERT + Full Ontology (RawGate)",
    train_loader_pho,
    valid_loader_pho,
    feature_key="ontology_features",
    patience=3,
)

# ==================== NHÓM 2: VISOBERT ====================
# 4. ViSoBERT Baseline
m_viso_base = Model_Baseline(NUM_CLASSES, model_name=VISO_ID)
m_viso_base = train_experiment(
    m_viso_base,
    "4. ViSoBERT Baseline",
    train_loader_viso,
    valid_loader_viso,
    feature_key="lexicon_features",
    patience=3,
)

# 5. ViSoBERT + VnEmoLex Concat (Doãn & Lưu, 2022)
m_viso_lex = Model_VnEmoLex_Concat(NUM_CLASSES, model_name=VISO_ID)
m_viso_lex = train_experiment(
    m_viso_lex,
    "5. ViSoBERT + VnEmoLex (Doãn & Lưu, 2022)",
    train_loader_viso,
    valid_loader_viso,
    feature_key="lexicon_features",
    patience=3,
)

# 6. ViSoBERT + Full Ontology (RawGate)
m_viso_ont = Model_Ontology_RawGate(NUM_CLASSES, model_name=VISO_ID)
m_viso_ont = train_experiment(
    m_viso_ont,
    "6. ViSoBERT + Full Ontology (RawGate)",
    train_loader_viso,
    valid_loader_viso,
    feature_key="ontology_features",
    patience=3,
)

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 1. PhoBERT Baseline


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

...

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 5. ViSoBERT + VnEmoLex (Doãn & Lưu, 2022)
Epoch 01 | Loss: 0.6756 | Val F1: 0.7420 | Val Acc: 0.8629 (New Best)
Epoch 02 | Loss: 0.4317 | Val F1: 0.8121 | Val Acc: 0.9109 (New Best)
Epoch 03 | Loss: 0.3154 | Val F1: 0.8017 | Val Acc: 0.9141 (Patience: 1/3)
Epoch 04 | Loss: 0.1951 | Val F1: 0.7831 | Val Acc: 0.9160 (Patience: 2/3)
Epoch 05 | Loss: 0.1387 | Val F1: 0.8012 | Val Acc: 0.9179 (Patience: 3/3)
Early stopping triggered after 5 epochs.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 6. ViSoBERT + Full Ontology (RawGate)
Epoch 01 | Loss: 0.6608 | Val F1: 0.7268 | Val Acc: 0.8572 (New Best)
Epoch 02 | Loss: 0.4460 | Val F1: 0.7820 | Val Acc: 0.9210 (New Best)
Epoch 03 | Loss: 0.3084 | Val F1: 0.7951 | Val Acc: 0.9191 (New Best)
Epoch 04 | Loss: 0.1877 | Val F1: 0.7993 | Val Acc: 0.9223 (New Best)
Epoch 05 | Loss: 0.1210 | Val F1: 0.7904 | Val Acc: 0.9210 (Patience: 1/3)
Epoch 06 | Loss: 0.0745 | Val F1: 0.7975 | Val Acc: 0.9236 (Patience: 2/3)
Epoch 07 | Loss: 0.0669 | Val F1: 0.8005 | Val Acc: 0.9242 (New Best)
Epoch 08 | Loss: 0.0432 | Val F1: 0.8071 | Val Acc: 0.9204 (New Best)
Epoch 09 | Loss: 0.0275 | Val F1: 0.7887 | Val Acc: 0.9204 (Patience: 1/3)
Epoch 10 | Loss: 0.0439 | Val F1: 0.7898 | Val Acc: 0.9141 (Patience: 2/3)
Epoch 11 | Loss: 0.0427 | Val F1: 0.8211 | Val Acc: 0.9299 (New Best)
Epoch 12 | Loss: 0.0302 | Val F1: 0.8079 | Val Acc: 0.9217 (Patience: 1/3)
Epoch 13 | Loss: 0.0244 | Val F1: 0.7960 | Val Acc: 0.9210 (Patience: 2/3)
Epoch

# 8. Final Benchmark Evaluation & Classification Reports
Evaluates all trained architectures on the test set and outputs macro F1, overall accuracy, and class-level classification metrics for OCA.

In [9]:
def get_predictions(model, loader, feature_key="lexicon_features"):
    """Extract ground truth and model predictions on a dataset loader."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            feat = (
                batch[feature_key].to(device) if feature_key in batch else None
            )

            outputs = model(ids, mask, feat)
            _, pr = torch.max(outputs, 1)
            preds.extend(pr.cpu().numpy())
            targets.extend(labels.cpu().numpy())
    return targets, preds


# 1. Thu thập dự đoán PhoBERT
y_true, p_pho_base = get_predictions(
    m_pho_base, test_loader_pho, feature_key="lexicon_features"
)
_, p_pho_lex = get_predictions(
    m_pho_lex, test_loader_pho, feature_key="lexicon_features"
)
_, p_pho_ont = get_predictions(
    m_pho_ont, test_loader_pho, feature_key="ontology_features"
)

# 2. Thu thập dự đoán ViSoBERT
_, p_viso_base = get_predictions(
    m_viso_base, test_loader_viso, feature_key="lexicon_features"
)
_, p_viso_lex = get_predictions(
    m_viso_lex, test_loader_viso, feature_key="lexicon_features"
)
_, p_viso_ont = get_predictions(
    m_viso_ont, test_loader_viso, feature_key="ontology_features"
)

models_preds = [
    ("PhoBERT", "Baseline (No External)", p_pho_base),
    ("PhoBERT", "VnEmoLex Concat (Doãn & Lưu, 2022)", p_pho_lex),
    ("PhoBERT", "Full Ontology RawGate (Ours)", p_pho_ont),
    ("ViSoBERT", "Baseline (No External)", p_viso_base),
    ("ViSoBERT", "VnEmoLex Concat (Doãn & Lưu, 2022)", p_viso_lex),
    ("ViSoBERT", "Full Ontology RawGate (Ours)", p_viso_ont),
]

summary_records = []
for backbone, method, preds in models_preds:
    summary_records.append(
        {
            "Backbone": backbone,
            "Knowledge Integration": method,
            "Accuracy": accuracy_score(y_true, preds),
            "Macro F1": f1_score(y_true, preds, average="macro"),
        }
    )

print("\n==================== TEST SET BENCHMARK RESULTS ====================")
df_benchmark = pd.DataFrame(summary_records)
print(df_benchmark.round(4).to_string(index=False))

print("\n========== REPORT: PhoBERT + VnEmoLex (Doãn & Lưu, 2022) ==========")
print(classification_report(y_true, p_pho_lex, target_names=CLASS_NAMES, digits=4))

print("\n========== REPORT: ViSoBERT + Full Ontology RawGate (Ours) ==========")
print(classification_report(y_true, p_viso_ont, target_names=CLASS_NAMES, digits=4))


==================== TEST SET BENCHMARK RESULTS ====================
Backbone              Knowledge Integration  Accuracy  Macro F1
 PhoBERT             Baseline (No External)    0.9337    0.8351
 PhoBERT VnEmoLex Concat (Doãn & Lưu, 2022)    0.9416    0.8470
 PhoBERT       Full Ontology RawGate (Ours)    0.9403    0.8447
ViSoBERT             Baseline (No External)    0.8970    0.7603
ViSoBERT VnEmoLex Concat (Doãn & Lưu, 2022)    0.8932    0.7654
ViSoBERT       Full Ontology RawGate (Ours)    0.9065    0.7674

========== REPORT: PhoBERT + VnEmoLex (Doãn & Lưu, 2022) ==========
              precision    recall  f1-score   support

    Negative     0.9435    0.9716    0.9573      1409
     Neutral     0.7068    0.5629    0.6267       167
    Positive     0.9595    0.9547    0.9571      1590

    accuracy                         0.9416      3166
   macro avg     0.8699    0.8297    0.8470      3166
weighted avg     0.9391    0.9416    0.9398      3166


========== REPORT: ViSoBERT + F

# 9. Neuro-Symbolic Rule-Based Post-Processing
Overrides uncertain neural predictions using high-confidence symbolic signals from the ontology feature representations.

In [10]:
class NeuroSymbolicReasoner:
    """Combines neural probability outputs with ontology polarities for rule-based overrides."""

    def __init__(
        self,
        model,
        engine,
        tokenizer,
        device,
        ontology_threshold=0.55,
        model_conf_limit=0.85,
        is_phobert=False,
    ):
        self.model = (
            model.module if isinstance(model, nn.DataParallel) else model
        )
        self.engine = engine
        self.tokenizer = tokenizer
        self.device = device
        self.ont_threshold = ontology_threshold
        self.model_conf_limit = model_conf_limit
        self.is_phobert = is_phobert
        self.idx2label = {0: "Negative", 1: "Neutral", 2: "Positive"}

    def infer(self, text):
        input_text = ViTokenizer.tokenize(text) if self.is_phobert else text
        inputs = self.tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            max_length=128,
            padding="max_length",
        )
        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)

        vec = self.engine.getvector(text)
        if isinstance(vec, tuple):
            vec = vec[0]
        ont_tensor = torch.tensor(np.array([vec]), dtype=torch.float).to(
            self.device
        )

        self.model.eval()
        with torch.no_grad():
            outputs = self.model(input_ids, attention_mask, ont_tensor)
            probs = F.softmax(outputs, dim=1)

        pred_idx = torch.argmax(probs).item()
        confidence = probs[0][pred_idx].item()
        neural_label = self.idx2label[pred_idx]

        # Chỉ số ontology: 19 là Positive, 20 là Negative
        pos_signal = vec[19]
        neg_signal = vec[20]
        final_label = neural_label

        # Cơ chế can thiệp: ghi đè khi dự đoán Neutral có độ tin cậy thấp nhưng ontology phát hiện phân cực mạnh
        if pred_idx == 1:
            if (
                neg_signal > self.ont_threshold
                and confidence < self.model_conf_limit
            ):
                final_label = "Negative"
            elif (
                pos_signal > self.ont_threshold
                and confidence < self.model_conf_limit
            ):
                final_label = "Positive"

        return neural_label, final_label


def run_rule_evaluation(model, tokenizer, is_phobert=False, name="ViSoBERT"):
    reasoner = NeuroSymbolicReasoner(
        model=model,
        engine=ontengine,
        tokenizer=tokenizer,
        device=device,
        ontology_threshold=0.55,
        model_conf_limit=0.85,
        is_phobert=is_phobert,
    )

    y_true_ns = []
    y_pred_base = []
    y_pred_neuro = []
    label_map = {"Negative": 0, "Neutral": 1, "Positive": 2}

    for _, row in tqdm(
        df_test.iterrows(), total=len(df_test), desc=f"Neuro-Symbolic ({name})"
    ):
        text = str(row["Sentence"])
        true_lbl = int(row["Sentiment"])

        base_lbl, neuro_lbl = reasoner.infer(text)
        y_true_ns.append(true_lbl)
        y_pred_base.append(label_map[base_lbl])
        y_pred_neuro.append(label_map[neuro_lbl])

    acc_base = accuracy_score(y_true_ns, y_pred_base)
    f1_base = f1_score(y_true_ns, y_pred_base, average="macro")
    acc_neuro = accuracy_score(y_true_ns, y_pred_neuro)
    f1_neuro = f1_score(y_true_ns, y_pred_neuro, average="macro")

    overridden_count = sum(
        1 for b, n in zip(y_pred_base, y_pred_neuro) if b != n
    )
    print(
        f"[{name}] Total overridden samples: {overridden_count}/{len(df_test)} ({overridden_count / len(df_test) * 100:.2f}%)"
    )

    return {
        "Backbone": name,
        "Baseline Acc": acc_base,
        "Neuro-Symbolic Acc": acc_neuro,
        "Delta Acc": acc_neuro - acc_base,
        "Baseline F1": f1_base,
        "Neuro-Symbolic F1": f1_neuro,
        "Delta F1": f1_neuro - f1_base,
    }


records = []
records.append(
    run_rule_evaluation(
        m_pho_base, tokenizer_phobert, is_phobert=True, name="PhoBERT"
    )
)
records.append(
    run_rule_evaluation(
        m_viso_base, tokenizer_visobert, is_phobert=False, name="ViSoBERT"
    )
)

df_rule_res = pd.DataFrame(records)
print("\n============ NEURO-SYMBOLIC RULE-BASED POST-PROCESSING ============")
print(df_rule_res.round(4).to_string(index=False))

Neuro-Symbolic (PhoBERT):   0%|          | 0/3166 [00:00<?, ?it/s]

[PhoBERT] Total overridden samples: 8/3166 (0.25%)


Neuro-Symbolic (ViSoBERT):   0%|          | 0/3166 [00:00<?, ?it/s]

[ViSoBERT] Total overridden samples: 19/3166 (0.60%)

============ NEURO-SYMBOLIC RULE-BASED POST-PROCESSING ============
Backbone  Baseline Acc  Neuro-Symbolic Acc  Delta Acc  Baseline F1  Neuro-Symbolic F1  Delta F1
 PhoBERT        0.9337              0.9340     0.0003       0.8351             0.8339   -0.0012
ViSoBERT        0.8970              0.8989     0.0019       0.7603             0.7610    0.0007


# 10. Statistical Significance Testing (Paired Permutation Test)
Calculates exact permutation p-values over 10,000 resamples to evaluate statistical significance ($p < 0.05$) across comparative models against OCA.

In [11]:
from src.significance_test import paired_permutation_test, macro_f1


comparisons = [
    # PhoBERT
    (
        "PhoBERT: Baseline vs VnEmoLex (Doãn & Lưu)",
        p_pho_base,
        p_pho_lex,
    ),
    (
        "PhoBERT: VnEmoLex vs Full Ontology",
        p_pho_lex,
        p_pho_ont,
    ),
    (
        "PhoBERT: Baseline vs Full Ontology",
        p_pho_base,
        p_pho_ont,
    ),
    # ViSoBERT
    (
        "ViSoBERT: Baseline vs VnEmoLex (Doãn & Lưu)",
        p_viso_base,
        p_viso_lex,
    ),
    (
        "ViSoBERT: VnEmoLex vs Full Ontology",
        p_viso_lex,
        p_viso_ont,
    ),
    (
        "ViSoBERT: Baseline vs Full Ontology",
        p_viso_base,
        p_viso_ont,
    ),
]

print(
    "\n============== STATISTICAL SIGNIFICANCE TESTS (10,000 ITERATIONS) =============="
)

results = []
for name, preds_a, preds_b in comparisons:
    f1_a, f1_b, diff_f1, p_val_f1 = paired_permutation_test(
        y_true, preds_a, preds_b, metric_func=macro_f1
    )
    acc_a, acc_b, diff_acc, p_val_acc = paired_permutation_test(
        y_true, preds_a, preds_b, metric_func=accuracy_score
    )

    sig_f1 = "Significant (p < 0.05)" if p_val_f1 < 0.05 else "Not Significant"
    sig_acc = (
        "Significant (p < 0.05)" if p_val_acc < 0.05 else "Not Significant"
    )

    results.append(
        {
            "Comparison": name,
            "F1 (Model A)": round(f1_a, 4),
            "F1 (Model B)": round(f1_b, 4),
            "Diff (F1)": round(diff_f1, 4),
            "p-val (F1)": round(p_val_f1, 5),
            "Sig (F1)": sig_f1,
            "p-val (Acc)": round(p_val_acc, 5),
            "Sig (Acc)": sig_acc,
        }
    )

df_perm = pd.DataFrame(results)
print(df_perm.to_string(index=False))
print(
    "\n================================================================================"
)


============== STATISTICAL SIGNIFICANCE TESTS (10,000 ITERATIONS) ==============
                                 Comparison  F1 (Model A)  F1 (Model B)  Diff (F1)  p-val (F1)        Sig (F1)  p-val (Acc)              Sig (Acc)
 PhoBERT: Baseline vs VnEmoLex (Doãn & Lưu)        0.8351        0.8470     0.0120     0.18318 Not Significant      0.00990 Significant (p < 0.05)
         PhoBERT: VnEmoLex vs Full Ontology        0.8470        0.8447     0.0023     0.78532 Not Significant      0.69983        Not Significant
         PhoBERT: Baseline vs Full Ontology        0.8351        0.8447     0.0096     0.31197 Not Significant      0.03540 Significant (p < 0.05)
ViSoBERT: Baseline vs VnEmoLex (Doãn & Lưu)        0.7603        0.7654     0.0051     0.62144 Not Significant      0.41626        Not Significant
        ViSoBERT: VnEmoLex vs Full Ontology        0.7654        0.7674     0.0020     0.85231 Not Significant      0.00210 Significant (p < 0.05)
        ViSoBERT: Baseline vs Full O